# Regex-only AIUB Faculty Scraper (Name / Email / Room No)

**NLP mini project** — collects the faculty **name, e-mail and room number** of the
*Faculty of Science & Technology* from
`https://www.aiub.edu/faculty-list/faculty-of-science--technology` using **regular expressions only**.

### Rules followed in this notebook

| Requirement | How it is handled |
|---|---|
| Only regular expressions | Every extraction, filter and cleanup is an `re` pattern. No BeautifulSoup, no `lxml`, no `json` module, no `pandas`. |
| Collect **name**, **email**, **room no** | Regex 6 pulls those keys out of the raw text; Regex 8-12 clean and validate them. |
| Small and runnable | Standard library only: `urllib`, `re`, `csv`, `collections`, `os`. |

### How the site is built (the interesting part)

The page renders its faculty cards with **JavaScript**, so the HTML the server returns
contains no names and no rooms. The notebook therefore:

1. downloads the page HTML and **proves** the data is not in it,
2. regex-matches the `<script>` file behind the page,
3. regex-matches the data-feed URL out of that script,
4. downloads the feed as **raw text** and parses every record with regex,
5. cleans the three fields and writes `output/fst_faculty.csv`.

Run all cells top-to-bottom (Restart & Run All).

In [1]:
# ---------------------------------------------------------------
# Standard library only.
# ---------------------------------------------------------------
import re
import csv
import os
import collections
import urllib.request

PAGE_URL = ("https://www.aiub.edu/faculty-list/faculty-of-science--technology"
            "?faculty=FACULTY+OF+SCIENCE+%26+TECHNOLOGY")
BASE_URL = "https://www.aiub.edu"
OUT_DIR = "output"

os.makedirs(OUT_DIR, exist_ok=True)
print("notebook folder:", os.getcwd())

notebook folder: /Users/roybipro/Developer/JNBook/NLP/Faculty-Data


## Step 1 — Download the page and check what is actually in the HTML

In [2]:
def fetch(url):
    # Download a URL and return the body as plain text. Nothing is parsed here.
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8", "replace")

# Regex 1: anything that looks like an e-mail address
LOOKS_LIKE_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w-]+(?:\.[\w-]+)*")
# Regex 2: the word 'room' / 'Room No' anywhere in the document
ROOM_WORD = re.compile(r"room\s*(?:no|number|#)?\.?", re.I)

html = fetch(PAGE_URL)

print("HTML characters       :", len(html))
print("e-mail looking hits   :", LOOKS_LIKE_EMAIL.findall(html))
print("'room' hits in HTML   :", len(ROOM_WORD.findall(html)))
print("occurrences of 'RoomNo':", html.count("RoomNo"))

HTML characters       : 123174
e-mail looking hits   : ['arahman@aiub.edu', 'arahman@aiub.edu', 'info@aiub.edu']
'room' hits in HTML   : 0
occurrences of 'RoomNo': 0


Confirmed: the served HTML is only the site shell plus menu — **no room numbers at all**.
The cards are injected by JavaScript, so the next job is to find *where* that script loads
its data. Everything below is discovered with regex; nothing is hard-coded.

## Step 2 — Find the data feed with regex

In [3]:
# Regex 3: the src attribute of the employee-profile script tag
#   matches:  src="/Content/employee-profile/script.js?v=23"
SCRIPT_SRC = re.compile(r'src="(?P<src>[^"]*employee-profile/script\.js[^"]*)"')

# Regex 4: the URL inside the fetch("...") call of that script
#   matches:  fetch("/Files/Uploads/public-employee-profiles/employeeProfiles.json?v=1.0.11")
FETCH_CALL = re.compile(r'fetch\( *"(?P<url>/Files/Uploads/[^"]+\.json[^"]*)"')

script_path = SCRIPT_SRC.search(html).group("src")
script_text = fetch(BASE_URL + script_path)
feed_path = FETCH_CALL.search(script_text).group("url")
FEED_URL = BASE_URL + feed_path

print("page script :", script_path)
print("data feed   :", FEED_URL)

page script : /Content/employee-profile/script.js?v=23
data feed   : https://www.aiub.edu/Files/Uploads/public-employee-profiles/employeeProfiles.json?v=1.0.11


## Step 3 — Download the feed and keep it as raw text

The text is also snapshotted to `output/faculty_feed_raw.txt` so the input data can be
inspected or handed in with the report. Each run re-downloads a fresh copy.

In [4]:
raw_text = fetch(FEED_URL)

# Snapshot of the raw input (not used for parsing - parsing always reads raw_text)
RAW_SNAPSHOT = os.path.join(OUT_DIR, "faculty_feed_raw.txt")
with open(RAW_SNAPSHOT, "w", encoding="utf-8") as handle:
    handle.write(raw_text)

print("feed characters :", len(raw_text))
print("snapshot saved  :", os.path.abspath(RAW_SNAPSHOT))
print("first 300 chars:")
print(raw_text[:300])

feed characters : 294863
snapshot saved  : /Users/roybipro/Developer/JNBook/NLP/Faculty-Data/output/faculty_feed_raw.txt
first 300 chars:
{"EmployeeProfileLightList":[{"IsShowProfileDetails":true,"CvPersonal":{"ID":7256,"UserId":60989,"Name":"MD. ANWARUL KABIR","Email":"kabir@aiub.edu"},"Faculty":"FACULTY OF SCIENCE & TECHNOLOGY","Designation":"Faculty","Position":"ASSOCIATE PROFESSOR","PositionID":137,"HrDepartment":"DEPARTMENT OF CO


## Step 4 — Split the text into one record per employee

The feed arrives as one very long line. Every employee object starts with the same key, so
a **lookahead split** cuts the text right in front of each of those keys while keeping the
opening brace inside the record:

```python
RECORD_START = re.compile(r'(?=\{"IsShowProfileDetails")')
```

Two details worth remembering for a viva answer:

* `\{` escapes the brace, otherwise `{...}` would be read as a repeat operator.
* `(?=...)` is a zero-width lookahead: it marks the cut position but does not consume the
  brace, so each piece still begins with `{"IsShowProfileDetails"`.

After that the parser is one tiny pattern per field — `"Key":"value"` — where `[^"]*`
means "everything up to the next quote".

In [5]:
# Regex 5: the split point, i.e. the position just before every record
RECORD_START = re.compile(r'(?=\{"IsShowProfileDetails")')

records = [r for r in RECORD_START.split(raw_text)
           if r.startswith('{"IsShowProfileDetails')]

def field(record, key):
    # Regex 6:  "RoomNo":"817"   ->   817
    match = re.search('"' + key + '":"(?P<value>[^"]*)"', record)
    return match.group("value") if match else ""

KEYS = ("Name", "Email", "Faculty", "Designation", "Position",
        "HrDepartment", "RoomNo", "BuildingNo")

people = [{key.lower(): field(record, key) for key in KEYS} for record in records]

print("records found  :", len(records))
print("employees      :", len(people))
print("sample         :", people[0])
print("missing name   :", sum(1 for p in people if not p["name"]))
print("missing email  :", sum(1 for p in people if not p["email"]))
print("empty room no  :", sum(1 for p in people if not p["roomno"]))

records found  : 446
employees      : 446
sample         : {'name': 'MD. ANWARUL KABIR', 'email': 'kabir@aiub.edu', 'faculty': 'FACULTY OF SCIENCE & TECHNOLOGY', 'designation': 'Faculty', 'position': 'ASSOCIATE PROFESSOR', 'hrdepartment': 'DEPARTMENT OF COMPUTER SCIENCE', 'roomno': 'DNO -216', 'buildingno': 'D - Building'}
missing name   : 0
missing email  : 0
empty room no  : 38


## Step 5 — Keep only the Faculty of Science & Technology

The stored value is `FACULTY OF SCIENCE & TECHNOLOGY`, while the page URL spells the same
thing `FACULTY+OF+SCIENCE+%26+TECHNOLOGY`, and other AIUB pages write `Science and
Technology`. One regex covers all of those spellings and any letter case.

In [6]:
# Regex 7: 'SCIENCE & TECHNOLOGY' in any spelling of the connector
IN_SCIENCE_TECH = re.compile(r"SCIENCE\s*(?:&amp;|&|AND)\s*TECHNOLOGY", re.I)

fst = [p for p in people if IN_SCIENCE_TECH.search(p["faculty"])]

print("all faculties on the feed:")
for name, count in collections.Counter(p["faculty"] for p in people).most_common():
    print("   {:>3}  {}".format(count, name))
print()
print("records selected for this project:", len(fst))

all faculties on the feed:
   196  FACULTY OF SCIENCE & TECHNOLOGY
    97  FACULTY OF ENGINEERING
    72  FACULTY OF ARTS AND SOCIAL SCIENCES
    65  FACULTY OF BUSINESS ADMINISTRATION
    16  FACULTY OF HEALTH AND LIFE SCIENCES

records selected for this project: 196


## Step 6 — Clean the three target fields (the NLP part)

| Field | Problem in the raw data | Regex fix |
|---|---|---|
| name | `PROF. DR. KH. ABDUL MALEQUE` is ALL CAPS and carries honorifics | Regex 8 strips known titles, Regex 12 title-cases the rest. `Md.` is *kept* because in Bangladeshi names it belongs to the given name |
| email | the loose unanchored Regex 1 matches any address-shaped fragment, including ones inside links | Regex 9 is anchored and strict, so it can be used as a real validation check |
| room no | messy formats: `817`, `DNGA05`, `DNO -216`, `9210K`, `1122 (OR)` | Regex 10 splits block letters + digits + wing letter, Regex 11 captures bracketed notes separately |
| department / building / position | stored in ALL CAPS as `DEPARTMENT OF COMPUTER SCIENCE` | Regex 13 title-cases them but keeps connectors like `of` / `and` lowercase |

In [7]:
# Regex 8: a leading honorific token such as 'PROF.' or 'DR.'
LEAD_TITLE = re.compile(r"^(?P<title>[A-Za-z]+\.)\s+(?P<rest>.*)$")
KNOWN_TITLES = {"prof", "dr", "mr", "ms", "mst", "engr", "rev"}

# Regex 9: a strictly valid e-mail address, anchored at both ends
VALID_EMAIL = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*"
                         r"\.[A-Za-z]{2,}$")

# Regex 10: room code = optional block letters + digits + optional wing letter
#   'DNO -216' -> block=DNO room=216 | '9210K' -> room=9210 wing=K
ROOM_PARTS = re.compile(r"(?P<block>[A-Za-z]{1,6})?[\s\-]*(?P<room>\d{1,6})"
                        r"(?:[\s\-]*(?P<wing>[A-Za-z]))?", re.I)

# Regex 11: a bracketed note such as '(OR)' or '(on Study Leave)'
BRACKET_NOTE = re.compile(r"\( *[^)]* *\)")

def clean_name(text):
    titles = []
    while True:
        match = LEAD_TITLE.match(text)
        if not match or match.group("title").rstrip(".").lower() not in KNOWN_TITLES:
            break
        titles.append(match.group("title"))
        text = match.group("rest")
    # Regex 12: capitalise the first letter of every word
    body = re.sub(r"\b([a-z])", lambda m: m.group(1).upper(), text.strip().lower())
    return (" ".join(t.title() for t in titles) + " " + body).strip()

def clean_room(text):
    match = ROOM_PARTS.search(text or "")
    note = BRACKET_NOTE.search(text or "")
    if not match:
        return {"block": "", "room": "", "wing": "", "note": ""}
    return {"block": (match.group("block") or "").upper(),
            "room": match.group("room"),
            "wing": (match.group("wing") or "").upper(),
            "note": note.group(0) if note else ""}

# Regex 13: lower-case connector words inside a label ('Department Of X' -> 'Department of X')
SMALL_WORDS = re.compile(r"(?<= )\b(Of|And|The|For|In|On)\b")

def clean_label(text):
    return SMALL_WORDS.sub(lambda m: m.group(1).lower(),
                           re.sub(r"\s+", " ", text).strip().title())

rows = []
for p in fst:
    room = clean_room(p["roomno"])
    rows.append({
        "name": clean_name(p["name"]),
        "email": p["email"].strip().lower(),
        "room_no": p["roomno"].strip(),
        "room_parsed": room["block"] + room["room"] + room["wing"],
        "building": clean_label(p["buildingno"]),
        "position": clean_label(p["position"]),
        "department": clean_label(p["hrdepartment"]),
    })

print("cleaned rows:", len(rows))
print()
for a, b in [("MD. ANWARUL KABIR", "name"), ("PROF. DR. KH. ABDUL MALEQUE", "name"),
             ("DR. MD. ABDULLAH - AL - JUBAIR", "name"), ("DNO -216", "room"),
             ("9210K", "room"), ("1122 (OR)", "room"), ("", "room")]:
    out = clean_name(a) if b == "name" else clean_room(a)
    print("   {!r:<36} -> {!r}".format(a, out))

cleaned rows: 196

   'MD. ANWARUL KABIR'                  -> 'Md. Anwarul Kabir'
   'PROF. DR. KH. ABDUL MALEQUE'        -> 'Prof. Dr. Kh. Abdul Maleque'
   'DR. MD. ABDULLAH - AL - JUBAIR'     -> 'Dr. Md. Abdullah - Al - Jubair'
   'DNO -216'                           -> {'block': 'DNO', 'room': '216', 'wing': '', 'note': ''}
   '9210K'                              -> {'block': '', 'room': '9210', 'wing': 'K', 'note': ''}
   '1122 (OR)'                          -> {'block': '', 'room': '1122', 'wing': '', 'note': '(OR)'}
   ''                                   -> {'block': '', 'room': '', 'wing': '', 'note': ''}


## Step 7 — Sanity checks before saving

In [8]:
bad_emails = [r["email"] for r in rows if not VALID_EMAIL.match(r["email"])]
no_room = [r["name"] for r in rows if not r["room_no"]]
dupes = [e for e, c in collections.Counter(r["email"] for r in rows).items() if c > 1]

print("rows                       :", len(rows))
print("e-mails failing the pattern:", len(bad_emails), bad_emails)
print("duplicate e-mails          :", len(dupes), dupes)
print("rows without a room number :", len(no_room), no_room)
print("room codes parsed          :",
      sum(1 for r in rows if r["room_parsed"]), "/", len(rows))

rows                       : 196
e-mails failing the pattern: 0 []
duplicate e-mails          : 0 []
rows without a room number : 11 ['Prof. Dr. Dip Nandi', 'Shahrin Chowdhury', 'Md. Asiful Islam', 'Asma Fariha', 'Fahmida Alam', 'Dr. M M Manjurul Islam', 'Md Moman Ul Haque Khan', 'Tamanna Zaman Bristy', 'Kamrun Naher Koli', 'Sadman Rafi', 'Dr. Mousumi Bala']
room codes parsed          : 183 / 196


## Step 8 — Save to CSV and preview

In [9]:
CSV_PATH = os.path.join(OUT_DIR, "fst_faculty.csv")
COLUMNS = ("name", "email", "room_no", "room_parsed", "building",
           "position", "department")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

print("saved ->", os.path.abspath(CSV_PATH))
print()
print("{:<33} {:<30} {:<10}".format("NAME", "EMAIL", "ROOM NO"))
print("-" * 75)
for r in rows[:15]:
    print("{:<33} {:<30} {:<10}".format(r["name"][:33], r["email"][:30],
                                        r["room_no"] or "-"))
print("... and {} more rows in the CSV".format(max(0, len(rows) - 15)))

saved -> /Users/roybipro/Developer/JNBook/NLP/Faculty-Data/output/fst_faculty.csv

NAME                              EMAIL                          ROOM NO   
---------------------------------------------------------------------------
Md. Anwarul Kabir                 kabir@aiub.edu                 DNO -216  
Mashiour Rahman                   mashiour@aiub.edu              DNGA05    
Salma Begom                       salma@aiub.edu                 817       
Dr. Mohammed Jashim Uddin         drjashim@aiub.edu              1122      
Prof. Dr. Kh. Abdul Maleque       maleque@aiub.edu               DN0915    
Dr. Sarfuddin Ahmed Tarek         satarek@aiub.edu               DN0722    
Dr. Afroza Nahar                  afroza@aiub.edu                DN0113    
Dr. S. A. M. Manzur H. Khan       manzur@aiub.edu                C         
Dr. Farzana Sabeth                fsabeth@aiub.edu               DN0721    
Dr. S. Mosaddeq Ahmed             smahmed@aiub.edu               DN0816    
Danil

## Step 9 — Small text analysis over the collected data

Note: bracketed leave notes are stripped first (Regex 11), so `LECTURER (on Study Leave)`
is counted as `LECTURER` — otherwise the same job would appear as several different titles.

In [10]:
# Positions, ignoring bracketed leave notes (Regex 11 reused)
positions = collections.Counter(BRACKET_NOTE.sub("", r["position"]).strip().title()
                                for r in rows)

# Letter blocks that prefix a room code, e.g. DN / DNO / DNGA in DN0915, DNO -216
blocks = collections.Counter(
    m.group(0) for r in rows
    for m in [re.match(r"[A-Za-z]+", r["room_parsed"])] if m)

# Most common name words, from a regex word tokenizer, minus stop-word-ish tokens
stop = {"md", "dr", "prof", "mst", "khan", "miah", "begum", "binte", "ibn", "ahmed"}
words = collections.Counter(
    w.capitalize() for r in rows for w in re.findall(r"[A-Za-z']{3,}", r["name"])
    if w.lower() not in stop)

# How many user-ids are a single lowercase word (i.e. basically the first name)?
short_ids = sum(1 for r in rows if re.match(r"^[a-z]+$", r["email"].split("@")[0]))

print("FST faculty collected  :", len(rows))
print("with a room number     :", len(rows) - len(no_room))
print("e-mail domains         :", collections.Counter(
      r["email"].split("@")[1] for r in rows).most_common(3))
print("single-word user ids   :", short_ids, "/", len(rows))
print("positions              :", positions.most_common())
print("room blocks            :", blocks.most_common(6))
print("top name words         :", words.most_common(10))

FST faculty collected  : 196
with a room number     : 185
e-mail domains         : [('aiub.edu', 196)]
single-word user ids   : 104 / 196
positions              : [('Lecturer', 81), ('Assistant Professor', 67), ('Associate Professor', 26), ('Professor', 17), ('Senior Assistant Professor', 3), ('Senior Associate Professor', 1), ('Senior Lecturer', 1)]
room blocks            : [('DN', 98), ('DS', 11), ('DNGA', 9), ('D', 5), ('DNO', 3), ('DE', 3)]
top name words         : [('Islam', 19), ('Hasan', 12), ('Rahman', 8), ('Mohammad', 7), ('Alam', 6), ('Rifat', 5), ('Hossain', 5), ('Uddin', 4), ('Farzana', 4), ('Jahan', 4)]


## Notes, limits and next steps

**What the project demonstrates** — a complete scrape-and-clean pipeline built from about
thirteen regular expressions and no third-party library: lookahead splitting for records,
`"key":"value"` matching for fields, anchored patterns for validation, and pattern-driven
normalisation of names, labels and room codes.

**Known limits (worth stating in a report)**

* The parser assumes the feed keeps the `{"IsShowProfileDetails"` record prefix and the
  current key names. If AIUB changes the feed, Regex 5/6 need re-checking — that fragility
  is exactly the trade-off of regex-only parsing versus a JSON/HTML parser.
* A value containing an escaped quote (`\"`) would break `[^"]*`; none exist in the feed today.
* 11 of the 196 selected records genuinely have an empty room number. They are kept as
  blanks instead of being dropped.
* Codes such as `1122 (OR)` are ambiguous at the source, so the note stays in its own
  column and the normalised form is stored separately — nothing is invented.

**Easy extensions**

* All faculties at once: `re.findall(r'href="(/faculty-list/[a-z0-9\-]+)"', html)` gives the
  menu links, then loop the same pipeline over each one.
* Pull any other field with the same helper, e.g. `field(record, "Designation")`.
* Feed the cleaned names into tokenising, frequency lists or a tiny search index.